# 4. 세 모델 답 비교하기 — 원본 · 내 모델 · GPT
학습에 쓰지 않은 시험 문제(`test.jsonl`)를 세 모델에 똑같이 묻고 나란히 비교해요.

| 순서 | 하는 일 | 결과 |
|---|---|---|
| 1 | 설정 | — |
| 2 | 세 모델에 묻기 (실행할 때마다 새로 물어요, 1~2분) | `results/compare_날짜시각.json` |
| 3 | 표로 보기 · 자동 체크 | 화면 |
| 4 | 웹 비교표 | `compare_report.html` (브라우저로 열림) |
| 5 | (선택) 블라인드 평가 | `compare_blind.html` |

준비: Ollama에 원본 `qwen2.5:3b`와 내 모델 등록 (`setup_ollama.ps1`, README) · `.env`에 `OPENAI_API_KEY` · 같은 폴더에 `compare_tools.py`

In [ ]:
# ▶ 1. 설정 — 여기만 고치세요
import os
import pandas as pd
from dotenv import load_dotenv
from compare_tools import connect, ask_models, auto_summary, make_report

MY_MODEL = 'mbcsports'      # ollama list 에 나온 내 모델 이름
load_dotenv()               # 같은 폴더의 .env 에서 OPENAI_API_KEY 읽기
MODELS = connect(MY_MODEL, os.getenv('OPENAI_API_KEY'))
pd.set_option('display.max_colwidth', None)
print('API 키:', '준비됨' if os.getenv('OPENAI_API_KEY') else '없음 → gpt-4.1-mini 칸은 (키 없음)')

In [ ]:
# ▶ 2. 세 모델에 묻기
test = pd.read_json('test.jsonl', lines=True)
print('지시문:', test['instruction'][0])
answers = ask_models(MODELS, test, 'compare')

In [ ]:
# ▶ 3-1. 표로 보기 — 정답과 세 모델 답 전체
pd.DataFrame({'정답 리드': test['output'], **answers})

In [ ]:
# ▶ 3-2. 자동 체크 요약 — 형식만 봐요 (참고용)
auto_summary(test, answers)

In [ ]:
# ▶ 4. 웹 비교표 — 원본(회색) · 내 모델(초록) · GPT(파랑) 순서 고정
make_report(test, answers, 'compare_report.html', '같은 문제, 세 모델의 리드 비교')

In [ ]:
# ▶ 5. (선택) 블라인드 평가 — 이름을 가리고 문제마다 A·B·C로 섞기
#    고른 뒤 맨 아래 '🔑 정답지'를 열어 누가 누구였는지 확인해요
make_report(test, answers, 'compare_blind.html', '블라인드 평가: 누가 쓴 리드일까?', blind=True)

## 사람이 확인할 것 — 자동 체크가 못 보는 부분
- **사실이 본문과 맞나?** 본문에 없는 원인 · 전망 · 인용 출처를 지어내지 않았나
- **누가 한 말인가?** 인터뷰 발언을 SNS 글이라고 쓰는 식의 바꿔치기
- **말이 자연스러운가?** 어색한 표현, 잘못된 띄어쓰기

⚠ `results/`와 `*.html`에는 기사 내용이 들어 있어요. GitHub에 올리지 않아요.